In [1]:
# import io
from pathlib import Path
import ebooklib
from ebooklib import epub
from lxml import etree

In [2]:
book_path = "../books/金閣寺 (三島由紀夫) (Z-Library).epub"
book = epub.read_epub(book_path)
documents = [item for item in book.get_items() if item.get_type() == ebooklib.ITEM_DOCUMENT]
documents

In [19]:
XHTML_NC = "http://www.w3.org/1999/xhtml"
test_document = documents[7]
content = test_document.content
print(test_document.content.decode("utf-8")[:1000])

<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html>
<html
 xmlns="http://www.w3.org/1999/xhtml"
 xmlns:epub="http://www.idpf.org/2007/ops"
 xml:lang="ja"
 class="vrtl"
>
<head>
<meta charset="UTF-8"/>
<title>金閣寺</title>
<link rel="stylesheet" type="text/css" href="../style/book-style.css"/>


</head>

<body>
<div class="main">
<p><br/></p>
<p><br/></p>
<p>　　　第　二　章</p>
<p><br/></p>
<p><br/></p>
<p>　父の死によって、私の本当の少年時代は終るが、自分の少年時代に、まるきり人間的関心ともいうべきものの、欠けていたことに私は<ruby>愕<rt>おどろ</rt></ruby>くのである。そしてこの愕きは、父の死を自分が少しも悲しんでいないのを知るに及んで、愕きとも名付けようのない、或る無力な感懐になった。</p>
<p>　<ruby>駈<rt>か</rt></ruby>けつけたとき、父はすでに棺の中に横たわっていた。というのは、内浦まで徒歩で行って、そこから船を頼んで、浦づたいに<ruby>成生<rt>なりう</rt></ruby>へかえるには、丸一日かかったからである。季節は<ruby>梅雨<rt>つゆ</rt></ruby>入り前の、照りつける暑い毎日である。私が対面すると<ruby>匆々<rt>そうそう</rt></ruby>、<ruby>柩<rt>ひつぎ</rt></ruby>は荒涼たる<ruby>岬<rt>みさき</rt></ruby>の焼場に運ばれて、海のほとりで焼かれることになっていた。</p>
<p>　田舎の寺の住職の死というものは、異様なものである。適切すぎて、異様なのである。彼はいわば、その地方の精神的中心でもあり、<ruby>檀家<rt>だんか</rt></ruby>の人たちのそれぞれの生涯の後見人でもあり、彼等の死後を委託される者でもあった。その彼が寺

In [27]:
print(type(test_document.content))

<class 'bytes'>


In [ ]:
print(test_document.get_body_content())

In [ ]:
root = etree.HTML(test_document.content)
print(root.text)
# print(root.find("head").find("link").attrib)

In [ ]:
XHTML_NS = "http://www.w3.org/1999/xhtml"
NS = {"x": XHTML_NS}

def remove_rp_rt(root: etree._Element) -> etree._Element:
    for node in root.xpath(".//x:rt | .//x:rp", namespaces=NS):
        parent = node.getparent()
        if parent is not None:
            parent.remove(node)
    return root

def extract_ebook_text(xhtml: bytes | str) -> str:
    parser = etree.XMLParser(recover=True, huge_tree=True)
    if isinstance(xhtml, str):
        xhtml = xhtml.encode("utf-8", errors="ignore")
    root = etree.fromstring(xhtml, parser=parser)
    
    root = remove_rp_rt(root)
    ps = root.xpath(".//x:body//x:p", namespaces=NS)